# Cross-file EDA

Checks the connections between the files that are not covered in the per-file notebooks. links vs movies_metadata, credits and ratings is covered in `links.ipynb`, and credits vs movies_metadata in `credits.ipynb`.

## 1. Imports

In [1]:
import pandas as pd

In [2]:
root = '../data'

## 2. Helper functions

In [3]:
def compare_ids(a, b, name_a, name_b):
    # print how many unique ids are in one file but not in the other
    a, b = set(a.dropna()), set(b.dropna())
    print(f"Nr of ids in {name_a} that are not in {name_b}: {len(a - b)}")
    print(f"Nr of ids in {name_b} that are not in {name_a}: {len(b - a)}")

## 3. Load data

Only the id columns are read. Ids in movies_metadata that cannot be read as numbers come from the wrongfully split rows, as in `credits.ipynb`.

In [4]:
mm_ids = pd.to_numeric(pd.read_csv(root + '/movies_metadata.csv', usecols=['id'], dtype=str).id, errors='coerce')
credit_ids = pd.read_csv(root + '/credits.csv', usecols=['id']).id
keyword_ids = pd.read_csv(root + '/keywords.csv', usecols=['id']).id

links = pd.read_csv(root + '/links.csv', dtype={'tmdbId': 'Int64'})
links_small = pd.read_csv(root + '/links_small.csv', dtype={'tmdbId': 'Int64'})

ratings = pd.read_csv(root + '/ratings.csv', usecols=['userId', 'movieId'])
ratings_small = pd.read_csv(root + '/ratings_small.csv')

## 4. keywords vs movies_metadata and credits

In [5]:
compare_ids(keyword_ids, mm_ids, 'keywords', 'movies_metadata')
compare_ids(keyword_ids, credit_ids, 'keywords', 'credits')
compare_ids(keyword_ids, links.tmdbId, 'keywords', 'links')

Nr of ids in keywords that are not in movies_metadata: 0
Nr of ids in movies_metadata that are not in keywords: 1
Nr of ids in keywords that are not in credits: 0
Nr of ids in credits that are not in keywords: 0
Nr of ids in keywords that are not in links: 0
Nr of ids in links that are not in keywords: 162


In [10]:
# read a few extra columns from movies_metadata to see what the movies are
mm = pd.read_csv(root + '/movies_metadata.csv', usecols=['id', 'imdb_id', 'title', 'release_date', 'status', 'vote_count'], dtype=str)
mm['id'] = pd.to_numeric(mm.id, errors='coerce')

# the movie in movies_metadata that is not in keywords or credits
display(mm[mm.id.notna() & ~mm.id.isin(keyword_ids) & ~mm.id.isin(credit_ids)])

,id,imdb_id,release_date,status,title,vote_count
42883,401840.0,tt3291632,2017-05-30,Released,School's out,0


In [11]:
# tmdbIds in links that are not in keywords (and therefore not in credits)
missing = links[links.tmdbId.notna() & ~links.tmdbId.isin(keyword_ids)]
print(f"Nr of rows in links: {len(missing)}, nr of unique tmdbIds: {missing.tmdbId.nunique()}")

# do these movies exist in movies_metadata?
in_mm = missing.tmdbId.isin(mm.id)
print(f"Nr of these that are in movies_metadata: {in_mm.sum()}")
display(missing[in_mm])

# how many ratings do these movies have?
n_ratings = ratings.movieId.isin(missing.movieId).sum()
print(f"Nr of ratings for these movies: {n_ratings} ({n_ratings / len(ratings):.2%})")
display(missing.head(10))

Nr of rows in links: 162, nr of unique tmdbIds: 162
Nr of these that are in movies_metadata: 1


,movieId,imdbId,tmdbId
43250,169932,3291632,401840


Nr of ratings for these movies: 29208 (0.11%)


,movieId,imdbId,tmdbId
4114,4207,100232,12773
4474,4568,96913,17882
4974,5069,270933,68149
5113,5209,81809,24549
7416,7669,112130,164721
8812,26379,75520,2413
8855,26452,83387,82205
8973,26649,96639,13057
8994,26693,99864,2670
9187,27049,135659,15738


## 5. Duplicate ids across files

All three TMDB files have duplicate ids. Are the same movies duplicated in every file?

In [6]:
dup_mm = set(mm_ids[mm_ids.duplicated()].dropna())
dup_credits = set(credit_ids[credit_ids.duplicated()])
dup_keywords = set(keyword_ids[keyword_ids.duplicated()])

print(f"Duplicate ids in movies_metadata: {len(dup_mm)}, credits: {len(dup_credits)}, keywords: {len(dup_keywords)}")
print(f"Duplicated in movies_metadata and credits: {len(dup_mm & dup_credits)}")
print(f"Duplicated in movies_metadata and keywords: {len(dup_mm & dup_keywords)}")
print(f"Duplicated in all three: {len(dup_mm & dup_credits & dup_keywords)}")

Duplicate ids in movies_metadata: 29, credits: 43, keywords: 985
Duplicated in movies_metadata and credits: 29
Duplicated in movies_metadata and keywords: 29
Duplicated in all three: 29


## 6. links_small vs links

In [7]:
print(f"There are {links_small.shape[0]} rows in links_small")
print(f"Nr of movieIds in links_small that are not in links: {(~links_small.movieId.isin(links.movieId)).sum()}")

# does links_small map to the same tmdbId and imdbId as links?
merged = links_small.merge(links, on='movieId', suffixes=('_small', ''))
print(f"Nr of rows where tmdbId differs: {(merged.tmdbId_small.fillna(-1) != merged.tmdbId.fillna(-1)).sum()}")
print(f"Nr of rows where imdbId differs: {(merged.imdbId_small != merged.imdbId).sum()}")

There are 9125 rows in links_small
Nr of movieIds in links_small that are not in links: 10
Nr of rows where tmdbId differs: 0
Nr of rows where imdbId differs: 0


## 7. ratings_small vs ratings and links_small

In [8]:
compare_ids(ratings_small.movieId, links_small.movieId, 'ratings_small', 'links_small')

print(f"Nr of movies in ratings_small that are not in ratings: {(~ratings_small.movieId.drop_duplicates().isin(ratings.movieId)).sum()}")

# is ratings_small a subset of ratings, i.e. do the same (userId, movieId) pairs exist?
pairs = ratings_small[['userId', 'movieId']].merge(ratings, on=['userId', 'movieId'], how='left', indicator=True)
print(f"Nr of ratings in ratings_small with the same (userId, movieId) in ratings: {(pairs._merge == 'both').sum()} of {len(pairs)}")

Nr of ids in ratings_small that are not in links_small: 0
Nr of ids in links_small that are not in ratings_small: 59
Nr of movies in ratings_small that are not in ratings: 8
Nr of ratings in ratings_small with the same (userId, movieId) in ratings: 6361 of 100004


In [12]:
# movieIds in links_small that are not in links
missing_links = links_small[~links_small.movieId.isin(links.movieId)]
display(missing_links)
print(f"Nr of these that exist in movies_metadata: {missing_links.tmdbId.isin(mm_ids).sum()}")

# are the movies in ratings_small that are not in ratings the same movies?
missing_ratings = set(ratings_small.movieId[~ratings_small.movieId.isin(ratings.movieId)])
print(f"Nr of the {len(missing_ratings)} movies in ratings_small not in ratings that are among these: {len(missing_ratings & set(missing_links.movieId))}")

,movieId,imdbId,tmdbId
5173,7502,185906,191903
7527,77658,81846,205300
8045,94466,2085059,<NA>
8505,108548,898266,<NA>
8507,108583,72500,<NA>
8520,108979,213338,314040
9062,150856,5189670,374454
9118,162376,4574334,410612
9123,164977,27660,137608
9124,164979,3447228,410803


Nr of these that exist in movies_metadata: 0
Nr of the 8 movies in ratings_small not in ratings that are among these: 8


In [13]:
# how many ratings in ratings_small belong to these 10 movies?
affected = ratings_small.movieId.isin(missing_links.movieId)
print(f"Nr of ratings in ratings_small for these movies: {affected.sum()} ({affected.mean():.2%})")
display(ratings_small[affected].groupby('movieId').size().rename('n_ratings'))

Nr of ratings in ratings_small for these movies: 46 (0.05%)


movieId
7502      22
77658      6
94466      7
108548     1
108583     3
108979     5
150856     1
162376     1
Name: n_ratings, dtype: int64

## 8. Findings and decisions

| Finding | Decision |
|---|---|
|One movie in movies_metadata, School’s out (id 401840, released 2017-05-30, vote_count 0), is in neither keywords nor credits. |Keep the movie without cast, crew and keywords|
|162 unique tmdbIds in links are in neither keywords nor credits. One of them is School’s out. The other 161 are not in movies_metadata either, so nothing is known about them. They have 29,208 ratings (0.11 %). Together with the 219 links without tmdbId, these are the 380 movies that cannot be joined with movies_metadata. | Action stated in links.ipynb|
|All 29 duplicated ids in movies_metadata are also duplicated in both credits and keywords. Credits has 14 and keywords 956 additional duplicated ids. A possible explanation is that the duplicates were created before the data was split into files. |Remove duplicate ids in all three files |
|links_small maps every shared movieId to the same tmdbId and imdbId as links, but has 10 movieIds that are not in links. None of these exist in movies_metadata (3 have no tmdbId), so nothing is known about them. The 8 movies in ratings_small that are not in ratings are all among these 10, and have 46 ratings (0.05 %). |Drop these 10 movies, since they cannot be linked to any movie information. Use links.csv as the mapping, as links_small adds nothing else. If ratings_small is used, this means dropping 46 ratings|